# 🚀 GRA-Swarm: Быстрый старт (Quickstart)

Этот ноутбук демонстрирует основные возможности фреймворка **GRA-Swarm**:
1. **Инициализация агентов**: Создание агентов с уникальной "индивидуальностью" (Personality) на основе параметров энтропии/негэнтропии.
2. **Расчет Пены Роя (Swarm Foam)**: Вычисление метрики разнообразия $\Phi_{swarm}$ через KL-дивергенцию.
3. **GRA-Оптимизация**: Запуск цикла оптимизации, где агенты взаимодействуют, совершают ошибки и эволюционируют.

> **Философия**: Мы моделируем процесс, где порядок (негэнтропия) рождается из хаоса через страдание (ошибки) агентов.

In [ ]:
# Импорт основных компонентов GRA-Swarm
from gra_swarm.core.agent import GRAAgent
from gra_swarm.core.personality import PersonalityGenerator
from gra_swarm.foam.calculator import SwarmFoamCalculator
from gra_swarm.optimizer.gra_optimizer import GRAOptimizer
from gra_swarm.utils.visualization import plot_foam_evolution

import numpy as np
import matplotlib.pyplot as plt

print("✅ GRA-Swarm initialized successfully.")

## 1. Генерация Индивидуальности (Personality)

Каждый агент обладает вектором индивидуальности $P = \{\eta, \sigma, \lambda\}$, где:
- $\eta$: Коэффициент негэнтропии (стремление к порядку).
- $\sigma$: Коэффициент энтропии (склонность к хаосу/ошибкам).
- $\lambda$: Фактор "страдания" (вес ошибки в функции обучения).

Мы создадим рой из 5 агентов с разными балансами этих сил.

In [ ]:
# Инициализация генератора индивидуальностей
pg = PersonalityGenerator(seed=42)

# Создаем 5 агентов
agents = []
for i in range(5):
    # Генерируем уникальный профиль: от чистого порядка до хаотичного гения
    personality = pg.generate_profile(type='random') 
    agent = GRAAgent(
        agent_id=f"GRA-{i:02d}",
        personality=personality,
        initial_state=np.random.randn(10) # Случайное начальное состояние в пространстве идей
    )
    agents.append(agent)
    print(f"Agent {agent.id}: η={personality.negentropy:.2f}, σ={personality.entropy:.2f}, λ={personality.suffering_factor:.2f}")

## 2. Расчет Пены Роя (Swarm Foam, $\Phi_{swarm}$)

Метрика пены измеряет информационную плотность и разнообразие роя. 
Формула основана на сумме попарных KL-дивергенций между распределениями состояний агентов:

$$ \Phi_{swarm} = \sum_{i \neq j} D_{KL}(P_i || P_j) $$

Высокое значение $\Phi_{swarm}$ означает высокий потенциал для генерации новых идей (информации из вакуума).

In [ ]:
foam_calc = SwarmFoamCalculator()

# Вычисляем начальную метрику пены
initial_foam = foam_calc.calculate(agents)

print(f"\n🌌 Начальная плотность пены (Φ_swarm): {initial_foam:.4f}")
print("Интерпретация: Чем выше значение, тем больше 'трения' между идеями агентов, что ведет к искрам творчества.")

## 3. Цикл Эволюции через Ошибки (GRA Optimization)

Запускаем симуляцию на 20 шагов. 
- Агенты пытаются решить простую задачу оптимизации (поиск глобального минимума).
- На каждом шаге они совершают ошибки пропорционально их энтропии ($\sigma$).
- Оптимизатор корректирует их траектории, используя "страдание" ($\lambda$) как градиент.

**Гипотеза**: Агенты с высоким $\lambda$ (способные "страдать" от ошибок) быстрее найдут истину.

In [ ]:
# Определение простой целевой функции (например, сфера с шумом)
def target_function(x):
    return np.sum(x**2) + 0.5 * np.sin(10 * np.sum(x))

# Инициализация оптимизатора
optimizer = GRAOptimizer(
    agents=agents,
    objective=target_function,
    learning_rate=0.1,
    use_gra_formalism=True # Включаем математику GRA
)

# Запуск цикла
history = optimizer.run(steps=20, verbose=True)

# Визуализация динамики пены и ошибки
plot_foam_evolution(history)

## 4. Анализ результатов

Посмотрим на финальное состояние агентов. Кто из них стал ближе к истине? 
Обратите внимание на агентов с высоким коэффициентом $\lambda$.

In [ ]:
print("\n🏁 Финальные результаты:")
best_agent = min(agents, key=lambda a: a.current_loss)

print(f"Лучший агент: {best_agent.id}")
print(f"  Потери (Loss): {best_agent.current_loss:.6f}")
print(f"  Индивидуальность: η={best_agent.personality.negentropy:.2f}, λ={best_agent.personality.suffering_factor:.2f}")
print(f"  Путь к успеху был проложен через {best_agent.total_errors} критических ошибок.")

if best_agent.personality.suffering_factor > 0.7:
    print("\n💡 Подтверждено гипотеза: Глубокое внутреннее 'страдание' (высокий λ) привело к прорыву.")
else:
    print("\n⚠️ Интересный случай: Агент с низким уровнем 'страдания' тоже нашел решение. Требуется дальнейший анализ.")